# Grad-CAM Explainability Demo for Seismic Event Detection

This notebook demonstrates the full Grad-CAM pipeline for seismic waveform interpretation.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import torch
import scipy.signal as sg

from src.model import UNet1D
from src.xai.gradcam import GradCAM1D, SeismicGradCAM
from src.visualization import SeismicVisualizer
from src.preprocessing import preprocess_signal

## 1. Load or Create Sample Data

In [ ]:
# Create synthetic seismic signal with arrival event
np.random.seed(42)
fs = 100  # sampling rate
duration = 10  # seconds
t = np.linspace(0, duration, int(fs * duration))

# Background noise
noise = 0.1 * np.random.randn(len(t))

# Simulated seismic arrival at t=5s
arrival_time = 5.0
arrival_idx = int(arrival_time * fs)
arrival = np.exp(-0.5 * ((t - arrival_time) / 0.3)**2) * np.sin(2 * np.pi * 5 * (t - arrival_time))

# Combine
waveform = noise + arrival

print(f'Waveform shape: {waveform.shape}')
print(f'Sampling rate: {fs} Hz')
print(f'True arrival at: {arrival_time}s (index {arrival_idx})')

## 2. Preprocess Signal

In [ ]:
processed = preprocess_signal(
    waveform,
    fs,
    lowcut=0.5,
    highcut=20.0,
    window_size_sec=1.0
)

print(f'Processed shape: {processed.shape}')

plt.figure(figsize=(12, 3))
plt.imshow(processed, aspect='auto', cmap='viridis')
plt.xlabel('Time samples')
plt.ylabel('IMF index')
plt.title('Preprocessed IMF Energy')
plt.colorbar()
plt.tight_layout()
plt.show()

## 3. Load Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = UNet1D(in_channels=5)
model_path = '../outputs/checkpoints/best_model.pth'

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print('Loaded pretrained model')
else:
    print('Using randomly initialized model (demo only)')

model = model.to(device)
model.eval()

## 4. Run Inference

In [ ]:
x = torch.tensor(processed[np.newaxis, :, :], dtype=torch.float32).to(device)

with torch.no_grad():
    pred = model(x).cpu().numpy().reshape(-1)

# Interpolate prediction to match original time axis
pred_interp = np.interp(t, np.linspace(t[0], t[-1], len(pred)), pred)

print(f'Prediction shape: {pred_interp.shape}')
print(f'Max confidence: {pred_interp.max():.4f}')

## 5. Generate Grad-CAM

In [ ]:
# Target the last encoder layer for Grad-CAM
target_layer = model.encoder3.se.fc

gradcam_generator = SeismicGradCAM(model, target_layer)
cam = gradcam_generator.compute(x, original_length=len(t))

print(f'CAM shape: {cam.shape}')
print(f'CAM range: [{cam.min():.4f}, {cam.max():.4f}]')

## 6. Core Visualization: Waveform + Prediction + Grad-CAM

In [ ]:
viz = SeismicVisualizer(save_dir='../outputs/plots')

viz.plot_waveform_prediction_gradcam(
    time=t,
    waveform=waveform,
    prediction=pred_interp,
    gradcam=cam,
    true_arrival_idx=arrival_idx,
    title='Grad-CAM Seismic Event Detection Demo',
    save_path='../outputs/plots/gradcam_demo_core.png'
)

## 7. IMF-wise Attention Analysis

In [ ]:
viz.plot_imf_attention(
    time=t,
    imf_energy=processed,
    gradcam=cam,
    title='IMF Contribution to Detection',
    save_path='../outputs/plots/gradcam_imf_attention.png'
)

## 8. Event Localization Assessment

In [ ]:
pred_arrival_idx = np.argmax(pred_interp)
confidence = pred_interp[pred_arrival_idx]

viz.plot_arrival_localization(
    time=t,
    waveform=waveform,
    prediction=pred_interp,
    true_arrival_idx=arrival_idx,
    pred_arrival_idx=pred_arrival_idx,
    confidence=confidence,
    title='Event Localization Accuracy',
    save_path='../outputs/plots/gradcam_localization.png'
)

## 9. Multi-Layer Grad-CAM Analysis

In [ ]:
# Generate CAM for different layers
layer_cams = {}
layers = {
    'Encoder 1': model.encoder1,
    'Encoder 2': model.encoder2,
    'Bottleneck': model.bottleneck
}

for name, layer in layers.items():
    try:
        gc = SeismicGradCAM(model, layer)
        layer_cam = gc.compute(x, original_length=len(t))
        layer_cams[name] = layer_cam
    except Exception as e:
        print(f'Skipping {name}: {e}')

if layer_cams:
    viz.plot_multilayer_gradcam(
        time=t,
        waveform=waveform,
        layer_cams=layer_cams,
        title='Multi-Layer Attention Analysis',
        save_path='../outputs/plots/gradcam_multilayer.png'
    )

## Summary

This demo showed:
1. Waveform preprocessing with EMD
2. Model inference
3. Grad-CAM attention generation
4. Core visualization (waveform + prediction + attention)
5. IMF-wise attention analysis
6. Event localization assessment
7. Multi-layer Grad-CAM analysis

All figures saved to `outputs/plots/`.